# ScienceQA — Ensemble Inference (multi-adapter + TTA)

Loads multiple LoRA adapters trained by `overnight_train.py`, scores each on the test
set with choice-permutation TTA, **averages the per-letter logits across all
adapters and all permutations**, and writes a submission.

This is the morning-after notebook. Set `ADAPTER_DIRS` below to the Drive paths
listed in `OVERNIGHT_SUMMARY.json`.


In [ ]:
%pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.0.1 pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# 1. Define paths
# UPDATE THIS to match where the 'images' folder is in your Google Drive
drive_images_path = '/content/drive/MyDrive/images/test'

local_data_dir = '/content/data'
local_images_path = '/content/data/images/test'

# Ensure the local data directory exists (where the CSVs should be)
os.makedirs(local_data_dir, exist_ok=True)

# 2. Copy images to local Colab storage for faster training I/O
if not os.path.exists(local_images_path):
    print(f"Copying images from {drive_images_path} to {local_images_path}...")
    print("This might take a few minutes depending on the size, but drastically speeds up training.")
    !cp -r "{drive_images_path}" "{local_data_dir}/"
    print("Copy complete! ✅")
else:
    print("Images already exist locally! ✅")


Copying images from /content/drive/MyDrive/images/test to /content/data/images/test...
This might take a few minutes depending on the size, but drastically speeds up training.
Copy complete! ✅


In [ ]:
import os, json, gc, random
from pathlib import Path
from typing import List, Optional, Dict, Any
import numpy as np
import pandas as pd
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ===== EDIT =====================================================
DATA_DIR = Path("/content/data")
ADAPTER_DIRS = [
    Path("/content/drive/MyDrive/scienceqa/outputs_train/adapter"),                    # Run A (384px, no CoT)
    Path("/content/drive/MyDrive/scienceqa_run_final_overnight/seed7_res512"),         # Run B (512px, no CoT)
    Path("/content/drive/MyDrive/scienceqa_run_final_overnight/seed21_res512_CoT"),    # Run C (512px, CoT)
]
IMG_SIZES = [384, 512, 512]
OUT_CSV = Path("/content/drive/MyDrive/scienceqa/submission_may3_midnight.csv")
# Per-adapter image size — must match what each was trained on
# ================================================================

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
TTA_PERMUTATIONS = 8         # per adapter
EVAL_BATCH = 64
MAX_CONTEXT_CHARS = 1200

assert len(ADAPTER_DIRS) == len(IMG_SIZES), "one img_size per adapter"
for d in ADAPTER_DIRS:
    assert d.exists(), f"missing adapter: {d}"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("adapters to ensemble:", len(ADAPTER_DIRS))

adapters to ensemble: 3


In [ ]:
# Load Drive

test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(json.loads)
print("test rows:", len(test_df))

test rows: 1008


In [ ]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def _truncate(s, n):
    s = str(s).strip()
    return s if len(s) <= n else s[:n-1] + "…"

def build_user_text(row, choices):
    parts = []
    lec = row.get("lecture")
    if isinstance(lec,str) and lec.strip(): parts.append(_truncate(lec, MAX_CONTEXT_CHARS))
    hint = row.get("hint")
    if isinstance(hint,str) and hint.strip(): parts.append(_truncate(hint, MAX_CONTEXT_CHARS//2))
    ctx = "\n".join(parts)
    cl = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i,c in enumerate(choices))
    t = ""
    if ctx: t += f"Context:\n{ctx}\n\n"
    t += f"Question: {row['question']}\n\nChoices:\n{cl}\n\nAnswer with a single letter only."
    return t

def build_messages(row, choices):
    return [{"role":"user","content":[
        {"type":"image"}, {"type":"text","text":build_user_text(row, choices)},
    ]}]

def load_image(rel, img_size):
    im = Image.open(DATA_DIR / rel).convert("RGB")
    w,h = im.size; s = img_size / max(w,h)
    if s < 1.0: im = im.resize((max(1,int(w*s)), max(1,int(h*s))), Image.BICUBIC)
    return im

# Pre-build row dicts for speed
test_rows = [{
    "id": r["id"], "image_path": r["image_path"], "question": r["question"],
    "choices": r["choices"], "lecture": r.get("lecture"), "hint": r.get("hint"),
} for _, r in test_df.iterrows()]
print("ready:", len(test_rows))

ready: 1008


In [ ]:
def score_with_adapter(adapter_dir: Path, img_size: int, tta_perms: int):
    """Return list of length len(test_rows); each element is np.ndarray of length
    num_choices_for_that_row containing AVERAGED logits across tta_perms permutations.
    """
    print(f"\n=== scoring with {adapter_dir.name}  (img_size={img_size}, TTA={tta_perms}) ===")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"

    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval(); model.config.use_cache = True

    def letter_id(L):
        ids = processor.tokenizer(L, add_special_tokens=False).input_ids
        return ids[-1] if len(ids) >= 1 else processor.tokenizer(" "+L, add_special_tokens=False).input_ids[-1]
    LIDS = {L: letter_id(L) for L in CHOICE_LETTERS}

    images = [load_image(r["image_path"], img_size) for r in test_rows]
    summed = [np.zeros(len(r["choices"]), dtype=np.float64) for r in test_rows]
    rng = np.random.RandomState(0)

    @torch.inference_mode()
    def score_one_pass(perms):
        for start in range(0, len(test_rows), EVAL_BATCH):
            sl = slice(start, min(start+EVAL_BATCH, len(test_rows)))
            rows_b = test_rows[sl]; imgs_b = images[sl]; perms_b = perms[sl]
            shuffled_choices = [[r["choices"][i] for i in p] for r,p in zip(rows_b, perms_b)]
            prompts = [processor.apply_chat_template(build_messages(r, ch), add_generation_prompt=True)
                       for r,ch in zip(rows_b, shuffled_choices)]
            inp = processor(text=prompts, images=[[im] for im in imgs_b], return_tensors="pt", padding=True)
            inp = {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in inp.items()}
            out = model(**inp); last = out.logits[:, -1, :]
            for i, p in enumerate(perms_b):
                ids = [LIDS[CHOICE_LETTERS[k]] for k in range(len(p))]
                row_logits = last[i, ids].float().cpu().numpy()
                gi = start + i
                for j, orig_idx in enumerate(p):
                    summed[gi][orig_idx] += float(row_logits[j])

    for k in range(tta_perms):
        perms = [(np.arange(len(r["choices"])) if k == 0 else rng.permutation(len(r["choices"])))
                 for r in test_rows]
        score_one_pass(perms)
        print(f"  pass {k+1}/{tta_perms} done")

    # free
    del model, base, processor; gc.collect(); torch.cuda.empty_cache()
    # average over passes for THIS adapter
    return [s / tta_perms for s in summed]

## Score each adapter, then average across adapters

In [ ]:
adapter_logits = []   # list (per adapter) of list (per row) of np.ndarray
for d, sz in zip(ADAPTER_DIRS, IMG_SIZES):
    adapter_logits.append(score_with_adapter(d, sz, TTA_PERMUTATIONS))

# Cross-adapter average
final = []
for i in range(len(test_rows)):
    arrs = [adapter_logits[a][i] for a in range(len(ADAPTER_DIRS))]
    final.append(np.mean(arrs, axis=0))

preds = [int(np.argmax(s)) for s in final]
print("preds done:", len(preds))


=== scoring with adapter  (img_size=384, TTA=8) ===
  pass 1/8 done
  pass 2/8 done
  pass 3/8 done
  pass 4/8 done
  pass 5/8 done
  pass 6/8 done
  pass 7/8 done
  pass 8/8 done

=== scoring with seed7_res512  (img_size=512, TTA=8) ===
  pass 1/8 done
  pass 2/8 done
  pass 3/8 done
  pass 4/8 done
  pass 5/8 done
  pass 6/8 done
  pass 7/8 done
  pass 8/8 done

=== scoring with seed21_res512_CoT  (img_size=512, TTA=8) ===
  pass 1/8 done
  pass 2/8 done
  pass 3/8 done
  pass 4/8 done
  pass 5/8 done
  pass 6/8 done
  pass 7/8 done
  pass 8/8 done
preds done: 1008


In [ ]:
sub = pd.DataFrame({"id":[r["id"] for r in test_rows], "answer": preds})
assert list(sub.columns) == ["id","answer"]
assert len(sub) == len(test_df)
assert set(sub["id"]) == set(test_df["id"])
assert sub["answer"].dtype.kind in ("i","u")
for r,p in zip(test_rows, preds):
    assert 0 <= p < len(r["choices"])

sub.to_csv(OUT_CSV, index=False)
print(f"✅ wrote {OUT_CSV}  ({len(sub)} rows)")
print("\nAnswer distribution:")
print(sub["answer"].value_counts().sort_index())

✅ wrote /content/drive/MyDrive/scienceqa/submission_may3_midnight.csv  (1008 rows)

Answer distribution:
answer
0    352
1    359
2    214
3     72
4     11
Name: count, dtype: int64


---
**Done.** Submit `submission.csv`.

Sanity checks:
- If predictions are nearly all `0`, the chat template / letter token IDs are off.
- If accuracy on train sample looks reasonable but submission scores low on Kaggle, you may have a path mismatch (check `image_path` resolves correctly).


In [ ]:
# ── 10) Disconnect runtime ───────────────────────────────────────────────
import time
print("Sleeping 60s to flush Drive…")

time.sleep(60)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("runtime.unassign failed:", e); os._exit(0)


Sleeping 60s to flush Drive…
